# Tool-use SFT — masked CE training

Fine-tunes the existing SFT/DPO checkpoint on the tool-use traces produced by `generate_tool_traces.ipynb` + `tokenize_tool_sft.ipynb`.

**Key differences vs `sft/sft_alpaca.ipynb`:**
- Loss is computed only on tokens where the loss mask is 1 (i.e. masking out `Question:` and `Result:` tokens).
- Conservative LR (1e-5) to avoid wiping out instruction-following from the previous stage.
- Configurable `START_FROM` so you can fine-tune from either DPO or SFT base.

**Before running:**
- Runtime → T4 GPU
- `tool_traces_tokenized.jsonl` must exist (run the previous two notebooks first)
- The chosen `START_FROM` checkpoint must exist on Drive

## 1. Setup + configuration

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, math, time, datetime, logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import drive

torch._logging.set_logs(inductor=logging.WARNING, dynamic=logging.WARNING)

drive.mount('/content/drive')

SPARKYLLM = '/content/drive/MyDrive/sparkyllm'
CHECKPOINT_DIR = os.path.join(SPARKYLLM, 'checkpoints')
MANIFEST_DIR   = os.path.join(SPARKYLLM, 'manifests')
DATA_DIR       = os.path.join(SPARKYLLM, 'datasets_sft', 'tool_traces')
TOKENIZED_FILE = os.path.join(DATA_DIR, 'tool_traces_tokenized.jsonl')
META_FILE      = os.path.join(DATA_DIR, 'tokenize_meta.json')
os.makedirs(MANIFEST_DIR, exist_ok=True)

# ---- Starting checkpoint selector ----
START_FROM = 'dpo'  # one of: 'dpo', 'sft'
START_CHECKPOINTS = {
    'dpo': 'gpt_medium_dpo_best.pth',
    'sft': 'gpt_medium_sft_best.pth',
}
assert START_FROM in START_CHECKPOINTS, f'unknown START_FROM: {START_FROM}'
START_CKPT = os.path.join(CHECKPOINT_DIR, START_CHECKPOINTS[START_FROM])
assert os.path.exists(START_CKPT), f'missing checkpoint: {START_CKPT}'

# ---- Output checkpoints ----
TOOLS_CKPT = os.path.join(CHECKPOINT_DIR, 'gpt_medium_tools.pth')      # in-progress save
BEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'gpt_medium_tools_best.pth')  # best by training loss

# ---- Architecture (must match pretrain) ----
BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64
FF_HIDDEN_DIM = 4 * EMBED_DIM

# ---- Training hyperparameters (conservative) ----
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4    # effective batch = 64
EPOCHS = 2
MAX_LR = 1e-5
MIN_LR = 1e-6
WARMUP_STEPS = 10       # was 100; with ~70 total optimizer steps, 100 left LR stuck in warmup
WEIGHT_DECAY = 0.05

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'START_FROM: {START_FROM} -> {START_CKPT}')

## 2. Load vocab metadata + tokenization meta

In [ ]:
with open(META_FILE) as f:
    tok_meta = json.load(f)
VOCAB_SIZE = int(tok_meta['vocab_size_padded'])
EOT_ID = int(tok_meta['eot_id'])
print(f'Vocab: {VOCAB_SIZE} | EOT: {EOT_ID}')
print(f'Examples in tokenized file: {tok_meta["num_examples"]}')

## 3. Model definition (matches pretrain)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.norm1(x)), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)), p=self.dropout, training=self.training)
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, dropout=0.05):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList([TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, dropout) for _ in range(NUM_LAYERS)])
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks: x = block(x)
        return self.lm_head(self.final_norm(x))

## 4. Tool-SFT dataset (with loss mask)

In [ ]:
class ToolSFTDataset(Dataset):
    def __init__(self, tokenized_path, block_size, eot_id):
        self.block_size = block_size
        self.eot_id = eot_id
        self.examples = []
        with open(tokenized_path) as f:
            for line in f:
                obj = json.loads(line)
                ids = obj['input_ids']
                mask = obj['loss_mask']
                if len(ids) > block_size + 1:
                    continue
                self.examples.append((ids, mask))
        print(f'Loaded {len(self.examples)} examples')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids, mask = self.examples[idx]
        # Pad to block_size + 1 (so x and y are block_size each)
        n = len(ids)
        if n < self.block_size + 1:
            pad = self.block_size + 1 - n
            ids = ids + [self.eot_id] * pad
            mask = mask + [0] * pad
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        m = torch.tensor(mask[1:], dtype=torch.float32)  # mask aligned with shifted targets
        return x, y, m

## 5. Load model + dataset

In [ ]:
model = SimpleGPT(VOCAB_SIZE).to(device)

print(f'Loading start checkpoint from {START_CKPT}...')
state_dict = torch.load(START_CKPT, map_location=device)
clean = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
msd = model.state_dict()
safe = {k: v for k, v in clean.items() if k in msd and v.shape == msd[k].shape}
model.load_state_dict(safe, strict=False)
print(f'Loaded {len(safe)}/{len(msd)} params')

dataset = ToolSFTDataset(TOKENIZED_FILE, BLOCK_SIZE, EOT_ID)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
print(f'Batches per epoch: {len(loader)}')

# Compile
print('Compiling model...')
model = torch.compile(model)

# Optimizer
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optimizer = optim.AdamW(
    [{'params': decay_params, 'weight_decay': WEIGHT_DECAY},
     {'params': nodecay_params, 'weight_decay': 0.0}],
    lr=MAX_LR, fused=True,
)

## 6. Training loop (masked CE)

In [ ]:
total_steps = (len(loader) * EPOCHS) // GRAD_ACCUM_STEPS
print(f'Total optimizer steps: {total_steps} | Warmup: {WARMUP_STEPS}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')

def get_lr(it, total_it):
    if it < WARMUP_STEPS:
        return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

best_loss = float('inf')
curr_step = 0
final_loss = None
train_start = time.time()

print(f'\nSTARTING TOOL SFT (BFloat16, masked CE, START_FROM={START_FROM})\n')

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_count = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}', dynamic_ncols=True, mininterval=1.0)
    for batch_idx, (xb, yb, mb) in enumerate(pbar):
        lr = get_lr(curr_step, total_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(xb)
            loss_per_tok = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                yb.reshape(-1),
                reduction='none',
            ).reshape(yb.shape)
            mask_sum = mb.sum().clamp(min=1)
            loss = (loss_per_tok * mb).sum() / mask_sum
            loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            curr_step += 1
            step_loss = loss.item() * GRAD_ACCUM_STEPS
            final_loss = step_loss
            epoch_loss_sum += step_loss
            epoch_count += 1
            elapsed = time.time() - train_start
            pbar.set_postfix(loss=f'{step_loss:.4f}', lr=f'{lr:.1e}', step=curr_step, mins=f'{elapsed/60:.1f}')
    avg_loss = epoch_loss_sum / max(1, epoch_count)
    print(f'Epoch {epoch} complete | avg_loss={avg_loss:.4f}')
    torch.save(model.state_dict(), TOOLS_CKPT)
    print(f'  Saved: {TOOLS_CKPT}')
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), BEST_CKPT)
        print(f'  New best! Saved: {BEST_CKPT}')

total_time = time.time() - train_start
print(f'\nTraining complete. {total_time/60:.1f} minutes, best_loss={best_loss:.4f}')

## 7. Save manifest

In [ ]:
manifest = {
    'stage': 'tool_sft',
    'created': datetime.datetime.now().isoformat(),
    'start_from': START_FROM,
    'start_checkpoint': os.path.basename(START_CKPT),
    'output_checkpoint': os.path.basename(BEST_CKPT),
    'config': {
        'block_size': BLOCK_SIZE,
        'embed_dim': EMBED_DIM,
        'num_layers': NUM_LAYERS,
        'batch_size': BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'epochs': EPOCHS,
        'max_lr': MAX_LR,
        'min_lr': MIN_LR,
        'warmup_steps': WARMUP_STEPS,
        'weight_decay': WEIGHT_DECAY,
        'precision': 'bfloat16',
        'loss': 'masked_ce',
    },
    'data': {
        'tokenized_file': TOKENIZED_FILE,
        'num_examples': len(dataset),
    },
    'results': {
        'best_loss': best_loss,
        'final_loss': final_loss,
        'total_optimizer_steps': curr_step,
        'total_minutes': round(total_time / 60, 2),
    },
}
manifest_path = os.path.join(MANIFEST_DIR, 'tool_sft_latest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest saved: {manifest_path}')
print(json.dumps(manifest, indent=2))